# YOLO v3b Threshold Sweep

Inference-only experiment for the fixed baseline:

- Dataset: `dataset_yolo_bbox_v3b_li_binary_medium`
- Model: YOLOv8n `best.pt`
- No training
- No dataset modification

The notebook evaluates confidence thresholds, NMS IoU thresholds, proposal coverage, TP/FN object sizes, and visual contact sheets.


## 1. Configuration


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/MataNerdy/Geodata_Archaeology_CV.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/kaggle/working/Geodata_Archaeology_CV")
PROJECT_DIR = REPO_DIR / "04_detection_yolo"

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORK_DATA_ROOT = Path("/kaggle/working/datasets")
OUTPUT_ROOT = Path("/kaggle/working/yolo_v3b_threshold_sweep")
ANALYSIS_DIR = OUTPUT_ROOT / "analysis"
WEIGHTS_WORK_DIR = OUTPUT_ROOT / "weights"

SOURCE_DATASET_DIR = Path("/kaggle/input/datasets/matanerdy/detection-dataset/dataset_yolo_bbox")
WEIGHTS_INPUT_PATH = Path("/kaggle/input/datasets/matanerdy/detection-dataset/best.pt")

DATASET_FOLDER = "dataset_yolo_bbox_v3b_li_binary_medium"
IMGSZ = 640
DEVICE = 0

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_WORK_DIR.mkdir(parents=True, exist_ok=True)
WORK_DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not str(WORK_DATA_ROOT.resolve()).startswith("/kaggle/working/"):
    raise ValueError(
        f"WORK_DATA_ROOT must be under /kaggle/working, got: {WORK_DATA_ROOT}. "
        "Kaggle input folders are read-only and must never be deleted or modified."
    )


## 2. Install Dependencies and Clone Repository


In [ ]:
import os
import shutil
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pandas", "pyyaml", "pillow", "matplotlib"],
    check=True,
)

os.chdir("/kaggle/working")
if REPO_DIR.exists():
    print("Removing existing working clone:", REPO_DIR)
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Project dir:", PROJECT_DIR)


## 3. Write Local Sweep Script


In [ ]:
SWEEP_SCRIPT = 'from __future__ import annotations\n\nimport argparse\nimport math\nimport os\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageFont\nfrom ultralytics import YOLO\n\nos.environ.setdefault("MPLBACKEND", "Agg")\nos.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")\n\ntry:\n    import matplotlib\n\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    MATPLOTLIB_ERROR = ""\nexcept Exception as exc:  # pragma: no cover - environment-specific fallback\n    plt = None\n    MATPLOTLIB_ERROR = repr(exc)\n\n\nCONF_SWEEP = [0.50, 0.25, 0.10, 0.05, 0.03, 0.01, 0.005, 0.003, 0.001]\nNMS_SWEEP = [0.40, 0.50, 0.60, 0.70, 0.80]\nVISUAL_CONFS = [0.25, 0.10, 0.05, 0.01]\nMATCH_IOU = 0.50\nCOVERAGE_IOU = 0.30\nSCRIPT_VERSION = "2026-06-14-path-order-fix"\n\n\n@dataclass(frozen=True)\nclass RunConfig:\n    name: str\n    conf: float\n    nms_iou: float\n    stage: str\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description="Run confidence/NMS threshold sweep for the v3b YOLO baseline."\n    )\n    parser.add_argument(\n        "--metadata",\n        type=Path,\n        default=Path("../datasets/dataset_yolo_bbox_v3b_li_binary_medium/metadata.csv"),\n    )\n    parser.add_argument(\n        "--weights",\n        type=Path,\n        default=Path(\n            "runs/yolo_baseline_comparison/runs/v3b_li_medium_yolov8n_img640/weights/best.pt"\n        ),\n    )\n    parser.add_argument(\n        "--out-dir",\n        type=Path,\n        default=Path("reports/threshold_sweep_v3b"),\n    )\n    parser.add_argument("--imgsz", type=int, default=640)\n    parser.add_argument("--max-visual-images", type=int, default=25)\n    parser.add_argument("--device", default=None)\n    return parser.parse_args()\n\n\ndef resolve_path(path_value: str | Path, base_dir: Path) -> Path:\n    path = Path(path_value)\n    if path.is_absolute():\n        return path\n    if path.exists():\n        return path\n    candidate = base_dir / path\n    if candidate.exists():\n        return candidate\n    return path\n\n\ndef xywhn_to_xyxy(row: pd.Series, image_size: int) -> tuple[float, float, float, float]:\n    xc = float(row["yolo_xc"]) * image_size\n    yc = float(row["yolo_yc"]) * image_size\n    w = float(row["yolo_w"]) * image_size\n    h = float(row["yolo_h"]) * image_size\n    return (xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2)\n\n\ndef box_iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:\n    if len(a) == 0 or len(b) == 0:\n        return np.zeros((len(a), len(b)), dtype=float)\n    x1 = np.maximum(a[:, None, 0], b[None, :, 0])\n    y1 = np.maximum(a[:, None, 1], b[None, :, 1])\n    x2 = np.minimum(a[:, None, 2], b[None, :, 2])\n    y2 = np.minimum(a[:, None, 3], b[None, :, 3])\n    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)\n    area_a = np.maximum(0, a[:, 2] - a[:, 0]) * np.maximum(0, a[:, 3] - a[:, 1])\n    area_b = np.maximum(0, b[:, 2] - b[:, 0]) * np.maximum(0, b[:, 3] - b[:, 1])\n    union = area_a[:, None] + area_b[None, :] - inter\n    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)\n\n\ndef load_validation_data(metadata_path: Path) -> tuple[pd.DataFrame, dict[str, pd.DataFrame], list[Path]]:\n    metadata_path = metadata_path.resolve()\n    project_dir = Path.cwd()\n    repo_root = project_dir.parent\n    df = pd.read_csv(metadata_path)\n    val_df = df[df["split"].astype(str).str.lower().eq("val")].copy()\n    if val_df.empty:\n        raise ValueError("No validation rows found in metadata.csv")\n\n    val_df["image_path"] = val_df["image"].apply(lambda p: resolve_path(p, project_dir))\n    val_df["image_key"] = val_df["image_path"].apply(lambda p: str(p.resolve()))\n\n    missing = sorted({str(p) for p in val_df["image_path"] if not Path(p).exists()})\n    if missing:\n        preview = "\\n".join(missing[:10])\n        raise FileNotFoundError(f"Validation images are missing. First paths:\\n{preview}")\n\n    gt_df = val_df[val_df["class_name"].notna()].copy()\n    gt_df["gt_id"] = np.arange(len(gt_df))\n    gt_df["bbox_xyxy"] = gt_df.apply(\n        lambda row: xywhn_to_xyxy(row, int(row.get("resize_to", 1024))), axis=1\n    )\n    gt_df["bbox_width_px"] = gt_df["bbox_x2_px"] - gt_df["bbox_x1_px"]\n    gt_df["bbox_height_px"] = gt_df["bbox_y2_px"] - gt_df["bbox_y1_px"]\n\n    gt_by_image = {\n        image_key: group.sort_values("gt_id").reset_index(drop=True)\n        for image_key, group in gt_df.groupby("image_key")\n    }\n    image_paths = [Path(p).resolve() for p in val_df.drop_duplicates("image_key")["image_path"]]\n    return val_df, gt_by_image, image_paths\n\n\ndef predict(model: YOLO, image_paths: list[Path], conf: float, nms_iou: float, imgsz: int, device: str | None) -> pd.DataFrame:\n    kwargs = {\n        "source": [str(p) for p in image_paths],\n        "imgsz": imgsz,\n        "conf": conf,\n        "iou": nms_iou,\n        "verbose": False,\n        "save": False,\n        "stream": False,\n    }\n    if device:\n        kwargs["device"] = device\n    results = model.predict(**kwargs)\n    rows: list[dict] = []\n    for result_idx, result in enumerate(results):\n        # Ultralytics may rewrite result.path to synthetic names such as image0.jpg\n        # when a Python list is used as input. The result order follows source order.\n        image_key = str(image_paths[result_idx].resolve())\n        if result.boxes is None or len(result.boxes) == 0:\n            continue\n        xyxy = result.boxes.xyxy.cpu().numpy()\n        confs = result.boxes.conf.cpu().numpy()\n        for idx, (box, score) in enumerate(zip(xyxy, confs)):\n            rows.append(\n                {\n                    "image_key": image_key,\n                    "pred_id": f"{Path(image_key).name}:{idx}",\n                    "x1": float(box[0]),\n                    "y1": float(box[1]),\n                    "x2": float(box[2]),\n                    "y2": float(box[3]),\n                    "confidence": float(score),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef match_predictions(\n    predictions: pd.DataFrame,\n    gt_by_image: dict[str, pd.DataFrame],\n    image_paths: list[Path],\n    low_conf_predictions: pd.DataFrame | None = None,\n) -> tuple[dict, pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    tp_rows: list[dict] = []\n    fp_rows: list[dict] = []\n    fn_rows: list[dict] = []\n    covered_gt = 0\n    total_gt = sum(len(group) for group in gt_by_image.values())\n\n    pred_by_image = {\n        image_key: group.sort_values("confidence", ascending=False).reset_index(drop=True)\n        for image_key, group in predictions.groupby("image_key")\n    }\n    low_pred_by_image = {}\n    if low_conf_predictions is not None and not low_conf_predictions.empty:\n        low_pred_by_image = {\n            image_key: group.sort_values("confidence", ascending=False).reset_index(drop=True)\n            for image_key, group in low_conf_predictions.groupby("image_key")\n        }\n\n    for image_path in image_paths:\n        image_key = str(image_path.resolve())\n        gt_group = gt_by_image.get(image_key, pd.DataFrame()).copy()\n        pred_group = pred_by_image.get(image_key, pd.DataFrame()).copy()\n\n        gt_boxes = np.array(gt_group["bbox_xyxy"].tolist(), dtype=float) if not gt_group.empty else np.empty((0, 4))\n        pred_boxes = (\n            pred_group[["x1", "y1", "x2", "y2"]].to_numpy(dtype=float)\n            if not pred_group.empty\n            else np.empty((0, 4))\n        )\n        ious = box_iou(pred_boxes, gt_boxes)\n\n        covered = set()\n        if len(pred_boxes) and len(gt_boxes):\n            for gt_idx in range(len(gt_boxes)):\n                if np.max(ious[:, gt_idx]) >= COVERAGE_IOU:\n                    covered.add(gt_idx)\n            covered_gt += len(covered)\n\n        matched_pred: set[int] = set()\n        matched_gt: set[int] = set()\n        candidates: list[tuple[float, int, int]] = []\n        if len(pred_boxes) and len(gt_boxes):\n            for pred_idx in range(len(pred_boxes)):\n                for gt_idx in range(len(gt_boxes)):\n                    if ious[pred_idx, gt_idx] >= MATCH_IOU:\n                        candidates.append((float(ious[pred_idx, gt_idx]), pred_idx, gt_idx))\n        for iou_value, pred_idx, gt_idx in sorted(candidates, reverse=True):\n            if pred_idx in matched_pred or gt_idx in matched_gt:\n                continue\n            matched_pred.add(pred_idx)\n            matched_gt.add(gt_idx)\n            gt_row = gt_group.iloc[gt_idx].to_dict()\n            pred_row = pred_group.iloc[pred_idx].to_dict()\n            tp_rows.append(\n                {\n                    **object_fields(gt_row),\n                    "image_key": image_key,\n                    "prediction_confidence": pred_row["confidence"],\n                    "prediction_iou": iou_value,\n                    "pred_x1": pred_row["x1"],\n                    "pred_y1": pred_row["y1"],\n                    "pred_x2": pred_row["x2"],\n                    "pred_y2": pred_row["y2"],\n                }\n            )\n\n        for pred_idx, pred_row in pred_group.iterrows():\n            if pred_idx in matched_pred:\n                continue\n            best_iou = float(np.max(ious[pred_idx])) if len(gt_boxes) else 0.0\n            fp_rows.append(\n                {\n                    "image_key": image_key,\n                    "confidence": pred_row["confidence"],\n                    "best_gt_iou": best_iou,\n                    "x1": pred_row["x1"],\n                    "y1": pred_row["y1"],\n                    "x2": pred_row["x2"],\n                    "y2": pred_row["y2"],\n                    "bbox_width_px": pred_row["x2"] - pred_row["x1"],\n                    "bbox_height_px": pred_row["y2"] - pred_row["y1"],\n                    "bbox_area_px": (pred_row["x2"] - pred_row["x1"]) * (pred_row["y2"] - pred_row["y1"]),\n                }\n            )\n\n        low_group = low_pred_by_image.get(image_key, pd.DataFrame()).copy()\n        low_boxes = (\n            low_group[["x1", "y1", "x2", "y2"]].to_numpy(dtype=float)\n            if not low_group.empty\n            else np.empty((0, 4))\n        )\n        low_ious = box_iou(low_boxes, gt_boxes)\n        for gt_idx, gt_row in gt_group.iterrows():\n            if gt_idx in matched_gt:\n                continue\n            base = object_fields(gt_row.to_dict())\n            best_low_iou = 0.0\n            best_low_conf = np.nan\n            if len(low_boxes):\n                best_idx = int(np.argmax(low_ious[:, gt_idx]))\n                best_low_iou = float(low_ious[best_idx, gt_idx])\n                best_low_conf = float(low_group.iloc[best_idx]["confidence"])\n            fn_rows.append(\n                {\n                    **base,\n                    "image_key": image_key,\n                    "best_low_conf_iou": best_low_iou,\n                    "best_low_confidence": best_low_conf,\n                }\n            )\n\n    tp = len(tp_rows)\n    fp = len(fp_rows)\n    fn = len(fn_rows)\n    precision = tp / (tp + fp) if tp + fp else 0.0\n    recall = tp / (tp + fn) if tp + fn else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    metrics = {\n        "TP": tp,\n        "FP": fp,\n        "FN": fn,\n        "Precision": precision,\n        "Recall": recall,\n        "F1": f1,\n        "covered_gt": covered_gt,\n        "total_gt": total_gt,\n        "coverage_rate": covered_gt / total_gt if total_gt else 0.0,\n    }\n    return metrics, pd.DataFrame(tp_rows), pd.DataFrame(fp_rows), pd.DataFrame(fn_rows)\n\n\ndef object_fields(row: dict) -> dict:\n    return {\n        "gt_id": row.get("gt_id"),\n        "source_class_name": row.get("source_class_name"),\n        "class_name": row.get("class_name"),\n        "region": row.get("region"),\n        "modality": row.get("modality"),\n        "bbox_area_px": row.get("bbox_area_px"),\n        "bbox_width_px": row.get("bbox_width_px"),\n        "bbox_height_px": row.get("bbox_height_px"),\n        "bbox_x1_px": row.get("bbox_x1_px"),\n        "bbox_y1_px": row.get("bbox_y1_px"),\n        "bbox_x2_px": row.get("bbox_x2_px"),\n        "bbox_y2_px": row.get("bbox_y2_px"),\n    }\n\n\ndef write_metrics_plot(df: pd.DataFrame, x_col: str, out_path: Path, title: str) -> None:\n    if plt is None:\n        write_placeholder_png(out_path, f"Plot skipped\\n{MATPLOTLIB_ERROR}")\n        return\n    fig, ax = plt.subplots(figsize=(8, 5))\n    for col in ["Precision", "Recall", "F1", "coverage_rate"]:\n        ax.plot(df[x_col], df[col], marker="o", label=col)\n    ax.set_title(title)\n    ax.set_xlabel(x_col)\n    ax.set_ylabel("score")\n    ax.grid(True, alpha=0.25)\n    ax.legend()\n    if x_col == "conf":\n        ax.invert_xaxis()\n    fig.tight_layout()\n    fig.savefig(out_path, dpi=180)\n    plt.close(fig)\n\n\ndef write_distribution_plots(tp: pd.DataFrame, fn: pd.DataFrame, out_dir: Path) -> None:\n    if plt is None:\n        for name in [\n            "tp_fn_bbox_area_px_distribution.png",\n            "tp_fn_bbox_width_px_distribution.png",\n            "tp_fn_bbox_height_px_distribution.png",\n            "tp_prediction_confidence_distribution.png",\n            "fn_best_low_confidence_distribution.png",\n        ]:\n            write_placeholder_png(out_dir / name, f"Plot skipped\\n{MATPLOTLIB_ERROR}")\n        return\n    dist = []\n    if not tp.empty:\n        t = tp.copy()\n        t["group"] = "TP"\n        dist.append(t)\n    if not fn.empty:\n        f = fn.copy()\n        f["group"] = "FN"\n        dist.append(f)\n    if not dist:\n        return\n    combined = pd.concat(dist, ignore_index=True)\n    for metric in ["bbox_area_px", "bbox_width_px", "bbox_height_px"]:\n        fig, ax = plt.subplots(figsize=(8, 5))\n        for group, group_df in combined.groupby("group"):\n            values = pd.to_numeric(group_df[metric], errors="coerce").dropna()\n            if values.empty:\n                continue\n            ax.hist(values, bins=24, alpha=0.55, label=group)\n        ax.set_title(f"{metric}: TP vs FN")\n        ax.set_xlabel(metric)\n        ax.set_ylabel("objects")\n        ax.grid(True, alpha=0.25)\n        ax.legend()\n        fig.tight_layout()\n        fig.savefig(out_dir / f"tp_fn_{metric}_distribution.png", dpi=180)\n        plt.close(fig)\n\n    if not tp.empty and "prediction_confidence" in tp.columns:\n        fig, ax = plt.subplots(figsize=(8, 5))\n        ax.hist(tp["prediction_confidence"].dropna(), bins=20, alpha=0.8)\n        ax.set_title("TP prediction confidence distribution")\n        ax.set_xlabel("confidence")\n        ax.set_ylabel("TP objects")\n        ax.grid(True, alpha=0.25)\n        fig.tight_layout()\n        fig.savefig(out_dir / "tp_prediction_confidence_distribution.png", dpi=180)\n        plt.close(fig)\n\n    if not fn.empty and "best_low_confidence" in fn.columns:\n        fig, ax = plt.subplots(figsize=(8, 5))\n        values = fn["best_low_confidence"].dropna()\n        if not values.empty:\n            ax.hist(values, bins=20, alpha=0.8)\n        ax.set_title("FN best low-confidence candidate distribution")\n        ax.set_xlabel("best candidate confidence at conf=0.01")\n        ax.set_ylabel("FN objects")\n        ax.grid(True, alpha=0.25)\n        fig.tight_layout()\n        fig.savefig(out_dir / "fn_best_low_confidence_distribution.png", dpi=180)\n        plt.close(fig)\n\n\ndef draw_boxes(\n    image_path: Path,\n    gt_rows: pd.DataFrame,\n    pred_rows: pd.DataFrame,\n    label: str,\n    max_size: int = 320,\n) -> Image.Image:\n    image = Image.open(image_path).convert("RGB")\n    original_w, original_h = image.size\n    scale = min(max_size / original_w, max_size / original_h)\n    new_size = (max(1, int(original_w * scale)), max(1, int(original_h * scale)))\n    image = image.resize(new_size)\n    draw = ImageDraw.Draw(image)\n    font = ImageFont.load_default()\n\n    for _, row in gt_rows.iterrows():\n        box = row["bbox_xyxy"] if "bbox_xyxy" in row else xywhn_to_xyxy(row, int(row.get("resize_to", original_w)))\n        scaled = tuple(float(v) * scale for v in box)\n        draw.rectangle(scaled, outline=(35, 220, 70), width=2)\n    for _, row in pred_rows.iterrows():\n        scaled = (\n            float(row["x1"]) * scale,\n            float(row["y1"]) * scale,\n            float(row["x2"]) * scale,\n            float(row["y2"]) * scale,\n        )\n        draw.rectangle(scaled, outline=(230, 45, 45), width=2)\n        draw.text((scaled[0] + 2, scaled[1] + 2), f"{row[\'confidence\']:.2f}", fill=(255, 235, 235), font=font)\n    draw.rectangle((0, 0, image.width, 16), fill=(0, 0, 0))\n    draw.text((3, 2), label[:48], fill=(255, 255, 255), font=font)\n    return image\n\n\ndef write_contact_sheet(images: list[Image.Image], out_path: Path, columns: int = 5, pad: int = 8) -> None:\n    if not images:\n        Image.new("RGB", (640, 120), "white").save(out_path)\n        return\n    cell_w = max(img.width for img in images)\n    cell_h = max(img.height for img in images)\n    rows = math.ceil(len(images) / columns)\n    sheet = Image.new("RGB", (columns * cell_w + (columns + 1) * pad, rows * cell_h + (rows + 1) * pad), "white")\n    for idx, img in enumerate(images):\n        x = pad + (idx % columns) * (cell_w + pad)\n        y = pad + (idx // columns) * (cell_h + pad)\n        sheet.paste(img, (x, y))\n    sheet.save(out_path, quality=92)\n\n\ndef write_placeholder_png(out_path: Path, text: str) -> None:\n    image = Image.new("RGB", (1000, 500), "white")\n    draw = ImageDraw.Draw(image)\n    font = ImageFont.load_default()\n    y = 24\n    for line in text.splitlines():\n        draw.text((24, y), line[:140], fill=(0, 0, 0), font=font)\n        y += 18\n    image.save(out_path)\n\n\ndef write_visuals(\n    conf: float,\n    predictions: pd.DataFrame,\n    fp: pd.DataFrame,\n    fn: pd.DataFrame,\n    gt_by_image: dict[str, pd.DataFrame],\n    image_paths: list[Path],\n    out_dir: Path,\n    max_images: int,\n) -> None:\n    pred_by_image = {k: g for k, g in predictions.groupby("image_key")}\n\n    pred_images = []\n    for image_path in image_paths[:max_images]:\n        image_key = str(image_path.resolve())\n        gt = gt_by_image.get(image_key, pd.DataFrame())\n        pred = pred_by_image.get(image_key, pd.DataFrame())\n        pred_images.append(draw_boxes(image_path, gt, pred, f"conf={conf:.2f} {image_path.name}"))\n    write_contact_sheet(pred_images, out_dir / f"conf_{conf:.2f}_predictions.jpg")\n\n    fp_images = []\n    for _, row in fp.head(max_images).iterrows():\n        image_path = Path(row["image_key"])\n        gt = gt_by_image.get(str(image_path.resolve()), pd.DataFrame())\n        pred = pd.DataFrame([row])\n        fp_images.append(draw_boxes(image_path, gt, pred, f"FP {row[\'confidence\']:.2f} {image_path.name}"))\n    write_contact_sheet(fp_images, out_dir / f"conf_{conf:.2f}_false_positives.jpg")\n\n    fn_images = []\n    for _, row in fn.head(max_images).iterrows():\n        image_path = Path(row["image_key"])\n        gt = gt_by_image.get(str(image_path.resolve()), pd.DataFrame())\n        one_gt = gt[gt["gt_id"].eq(row["gt_id"])] if not gt.empty and "gt_id" in gt else pd.DataFrame([row])\n        pred = pred_by_image.get(str(image_path.resolve()), pd.DataFrame())\n        fn_images.append(draw_boxes(image_path, one_gt, pred, f"FN {image_path.name}"))\n    write_contact_sheet(fn_images, out_dir / f"conf_{conf:.2f}_false_negatives.jpg")\n\n\ndef format_float(value: float) -> str:\n    return f"{value:.4f}"\n\n\ndef markdown_table(df: pd.DataFrame, floatfmt: str = ".4f") -> str:\n    if df.empty:\n        return "_No rows._"\n    formatted = df.copy()\n    for col in formatted.columns:\n        if pd.api.types.is_float_dtype(formatted[col]):\n            formatted[col] = formatted[col].map(lambda value: format(value, floatfmt))\n    formatted = formatted.fillna("")\n    headers = list(formatted.columns)\n    lines = [\n        "| " + " | ".join(headers) + " |",\n        "| " + " | ".join(["---"] * len(headers)) + " |",\n    ]\n    for _, row in formatted.iterrows():\n        lines.append("| " + " | ".join(str(row[col]) for col in headers) + " |")\n    return "\\n".join(lines)\n\n\ndef write_report(\n    out_path: Path,\n    confidence_df: pd.DataFrame,\n    nms_df: pd.DataFrame,\n    best_conf: float,\n    best_row: pd.Series,\n    best_nms_row: pd.Series,\n    tp: pd.DataFrame,\n    fp: pd.DataFrame,\n    fn: pd.DataFrame,\n) -> None:\n    low = confidence_df.sort_values("conf").iloc[0]\n    baseline = confidence_df[confidence_df["conf"].eq(0.25)]\n    baseline_row = baseline.iloc[0] if not baseline.empty else confidence_df.iloc[0]\n\n    recall_gain = float(low["Recall"] - baseline_row["Recall"])\n    fp_gain = int(low["FP"] - baseline_row["FP"])\n    covered_with_candidate = 0\n    if not fn.empty and "best_low_confidence" in fn.columns:\n        covered_with_candidate = int((fn["best_low_conf_iou"] >= COVERAGE_IOU).sum())\n\n    lines = [\n        "# v3b YOLO Threshold Sweep",\n        "",\n        "Inference-only analysis for `dataset_yolo_bbox_v3b_li_medium` using the existing YOLOv8n `best.pt`.",\n        "",\n        "## Confidence Sweep",\n        "",\n        markdown_table(confidence_df),\n        "",\n        "## NMS Sweep",\n        "",\n        markdown_table(nms_df),\n        "",\n        "## Selected Configurations",\n        "",\n        f"- Best confidence by F1 at NMS IoU 0.50: `conf={best_conf:.2f}`.",\n        f"- Best NMS setting for that confidence by F1: `iou={float(best_nms_row[\'nms_iou\']):.2f}`.",\n        f"- Best confidence row: Precision `{format_float(float(best_row[\'Precision\']))}`, Recall `{format_float(float(best_row[\'Recall\']))}`, F1 `{format_float(float(best_row[\'F1\']))}`, FP `{int(best_row[\'FP\'])}`, FN `{int(best_row[\'FN\'])}`.",\n        f"- Low-confidence `conf={float(low[\'conf\']):.2f}` row: Precision `{format_float(float(low[\'Precision\']))}`, Recall `{format_float(float(low[\'Recall\']))}`, coverage `{format_float(float(low[\'coverage_rate\']))}`, FP `{int(low[\'FP\'])}`.",\n        "",\n        "## TP / FN Object Size Summary",\n        "",\n        markdown_table(size_summary(tp, fn), floatfmt=".2f"),\n        "",\n        "## Interpretation",\n        "",\n        f"1. Recall change from `conf={float(baseline_row[\'conf\']):.2f}` to `conf={float(low[\'conf\']):.2f}` is `{recall_gain:+.4f}`, while FP changes by `{fp_gain:+d}`.",\n        f"2. Among FNs at the selected best-F1 configuration, `{covered_with_candidate}` have a low-confidence candidate with IoU >= {COVERAGE_IOU:.2f}.",\n        "3. If recall grows only slightly as confidence drops, the model is not just threshold-conservative: many GT objects are not proposed with sufficient spatial overlap.",\n        "4. If coverage is materially higher than strict recall, low-confidence inference can still be useful as a proposal generator before segmentation/manual review.",\n        "",\n        "## Generated Artifacts",\n        "",\n        "- `confidence_sweep_metrics.csv`",\n        "- `nms_sweep_metrics.csv`",\n        "- `all_sweep_metrics.csv`",\n        "- `tp_objects_best_f1.csv`",\n        "- `false_positives_best_f1.csv`",\n        "- `false_negatives_best_f1.csv`",\n        "- metric/distribution plots as PNG",\n        "- contact sheets for `conf=0.25`, `0.10`, `0.05`, `0.01`",\n        "",\n    ]\n    out_path.write_text("\\n".join(lines), encoding="utf-8")\n\n\ndef size_summary(tp: pd.DataFrame, fn: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    for group_name, df in [("TP", tp), ("FN", fn)]:\n        if df.empty:\n            continue\n        for metric in ["bbox_area_px", "bbox_width_px", "bbox_height_px"]:\n            values = pd.to_numeric(df[metric], errors="coerce").dropna()\n            if values.empty:\n                continue\n            rows.append(\n                {\n                    "group": group_name,\n                    "metric": metric,\n                    "count": len(values),\n                    "mean": values.mean(),\n                    "p25": values.quantile(0.25),\n                    "median": values.median(),\n                    "p75": values.quantile(0.75),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef main() -> None:\n    args = parse_args()\n    out_dir = args.out_dir\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    metadata_path = resolve_path(args.metadata, Path.cwd())\n    weights_path = resolve_path(args.weights, Path.cwd())\n    if not metadata_path.exists():\n        raise FileNotFoundError(metadata_path)\n    if not weights_path.exists():\n        raise FileNotFoundError(weights_path)\n\n    _, gt_by_image, image_paths = load_validation_data(metadata_path)\n    model = YOLO(str(weights_path))\n    print(f"Threshold sweep script version: {SCRIPT_VERSION}")\n    print(f"Weights: {weights_path}")\n    print(f"Validation images: {len(image_paths)}")\n    print(f"Validation GT objects: {sum(len(group) for group in gt_by_image.values())}")\n\n    prediction_cache: dict[tuple[float, float], pd.DataFrame] = {}\n    prediction_count_rows: list[dict] = []\n\n    def get_predictions(conf: float, nms_iou: float) -> pd.DataFrame:\n        key = (conf, nms_iou)\n        if key not in prediction_cache:\n            print(f"Predicting conf={conf:.2f}, nms_iou={nms_iou:.2f}")\n            prediction_cache[key] = predict(model, image_paths, conf, nms_iou, args.imgsz, args.device)\n            pred_df = prediction_cache[key]\n            prediction_count_rows.append(\n                {\n                    "conf": conf,\n                    "nms_iou": nms_iou,\n                    "predictions": len(pred_df),\n                    "images_with_predictions": pred_df["image_key"].nunique() if not pred_df.empty else 0,\n                    "max_confidence": pred_df["confidence"].max() if not pred_df.empty else np.nan,\n                }\n            )\n            print(\n                "Predictions:",\n                len(pred_df),\n                "images:",\n                prediction_count_rows[-1]["images_with_predictions"],\n                "max_conf:",\n                prediction_count_rows[-1]["max_confidence"],\n            )\n        return prediction_cache[key]\n\n    low_conf_predictions = get_predictions(min(CONF_SWEEP), 0.50)\n\n    all_rows = []\n    details: dict[str, tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]] = {}\n    for conf in CONF_SWEEP:\n        preds = get_predictions(conf, 0.50)\n        metrics, tp, fp, fn = match_predictions(preds, gt_by_image, image_paths, low_conf_predictions)\n        row = {"stage": "confidence", "conf": conf, "nms_iou": 0.50, **metrics}\n        all_rows.append(row)\n        details[f"conf_{conf:.2f}_iou_0.50"] = (preds, tp, fp, fn)\n\n    confidence_df = pd.DataFrame(all_rows).query("stage == \'confidence\'").sort_values("conf", ascending=False)\n    best_row = confidence_df.sort_values(["F1", "Recall", "Precision"], ascending=False).iloc[0]\n    best_conf = float(best_row["conf"])\n\n    nms_rows = []\n    for nms_iou in NMS_SWEEP:\n        preds = get_predictions(best_conf, nms_iou)\n        metrics, tp, fp, fn = match_predictions(preds, gt_by_image, image_paths, low_conf_predictions)\n        row = {"stage": "nms", "conf": best_conf, "nms_iou": nms_iou, **metrics}\n        nms_rows.append(row)\n        details[f"conf_{best_conf:.2f}_iou_{nms_iou:.2f}"] = (preds, tp, fp, fn)\n\n    nms_df = pd.DataFrame(nms_rows).sort_values("nms_iou")\n    all_df = pd.concat([confidence_df, nms_df], ignore_index=True)\n    best_nms_row = nms_df.sort_values(["F1", "Recall", "Precision"], ascending=False).iloc[0]\n\n    best_key = f"conf_{best_conf:.2f}_iou_{float(best_nms_row[\'nms_iou\']):.2f}"\n    _, best_tp, best_fp, best_fn = details[best_key]\n\n    confidence_df.to_csv(out_dir / "confidence_sweep_metrics.csv", index=False)\n    nms_df.to_csv(out_dir / "nms_sweep_metrics.csv", index=False)\n    all_df.to_csv(out_dir / "all_sweep_metrics.csv", index=False)\n    pd.DataFrame(prediction_count_rows).to_csv(out_dir / "prediction_counts.csv", index=False)\n    best_tp.to_csv(out_dir / "tp_objects_best_f1.csv", index=False)\n    best_fp.to_csv(out_dir / "false_positives_best_f1.csv", index=False)\n    best_fn.to_csv(out_dir / "false_negatives_best_f1.csv", index=False)\n\n    write_metrics_plot(confidence_df, "conf", out_dir / "confidence_sweep_metrics.png", "Confidence sweep at NMS IoU 0.50")\n    write_metrics_plot(nms_df, "nms_iou", out_dir / "nms_sweep_metrics.png", f"NMS sweep at conf={best_conf:.2f}")\n    write_distribution_plots(best_tp, best_fn, out_dir)\n\n    for conf in VISUAL_CONFS:\n        key = f"conf_{conf:.2f}_iou_0.50"\n        preds, _, fp, fn = details[key]\n        write_visuals(conf, preds, fp, fn, gt_by_image, image_paths, out_dir, args.max_visual_images)\n\n    write_report(\n        out_dir / "threshold_sweep_report.md",\n        confidence_df,\n        nms_df,\n        best_conf,\n        best_row,\n        best_nms_row,\n        best_tp,\n        best_fp,\n        best_fn,\n    )\n\n    print(f"Done. Report: {out_dir / \'threshold_sweep_report.md\'}")\n\n\nif __name__ == "__main__":\n    main()\n'

scripts_dir = PROJECT_DIR / "scripts"
scripts_dir.mkdir(parents=True, exist_ok=True)
sweep_script_path = scripts_dir / "sweep_v3b_thresholds.py"
sweep_script_path.write_text(SWEEP_SCRIPT, encoding="utf-8")
print("Sweep script:", sweep_script_path)


## 4. Locate Dataset


In [ ]:
import importlib.util
import shutil
import sys
import zipfile

import pandas as pd

def find_dataset_dir(folder_name: str) -> Path | None:
    for meta in KAGGLE_INPUT_ROOT.rglob("metadata.csv"):
        parent = meta.parent
        if parent.name == folder_name and (parent / "images").exists() and (parent / "labels").exists():
            return parent
    return None

def find_source_dataset() -> Path:
    if SOURCE_DATASET_DIR.exists() and (SOURCE_DATASET_DIR / "metadata.csv").exists():
        return SOURCE_DATASET_DIR
    found = find_dataset_dir("dataset_yolo_bbox")
    if found is not None:
        return found
    zip_candidates = sorted(KAGGLE_INPUT_ROOT.rglob("dataset_yolo_bbox.zip"))
    if zip_candidates:
        unzip_root = WORK_DATA_ROOT / "_source_unzipped"
        if unzip_root.exists():
            shutil.rmtree(unzip_root)
        unzip_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_candidates[0], "r") as zf:
            zf.extractall(unzip_root)
        found = next(unzip_root.rglob("metadata.csv")).parent
        if found.name == "dataset_yolo_bbox":
            return found
    raise FileNotFoundError("Attach a Kaggle input containing dataset_yolo_bbox/ or dataset_yolo_bbox.zip")

scripts_dir = PROJECT_DIR / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

prebuilt = find_dataset_dir(DATASET_FOLDER)
work_dir = WORK_DATA_ROOT / DATASET_FOLDER
if work_dir.exists():
    if not str(work_dir.resolve()).startswith("/kaggle/working/"):
        raise ValueError(f"Refusing to delete read-only or unsafe path: {work_dir}")
    shutil.rmtree(work_dir)

if prebuilt is not None:
    print("Copying prebuilt v3b dataset:", prebuilt)
    shutil.copytree(prebuilt, work_dir)
else:
    source_dataset = find_source_dataset()
    print("Building v3b from source dataset:", source_dataset)
    spec = importlib.util.spec_from_file_location("ablation", PROJECT_DIR / "scripts" / "build_dataset_ablation.py")
    if spec is None or spec.loader is None:
        raise ImportError("Could not load build_dataset_ablation.py")
    ablation = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = ablation
    spec.loader.exec_module(ablation)
    source_meta = ablation.read_source_metadata(source_dataset)
    version_by_name = {v.name: v for v in ablation.VERSIONS}
    ablation.build_dataset(
        source_dir=source_dataset,
        source_meta=source_meta,
        version=version_by_name["v3b_medium"],
        output_root=WORK_DATA_ROOT,
        overwrite=True,
        impact_rows=[],
    )

dataset_yaml = work_dir / "dataset.yaml"
dataset_yaml.write_text(
    f"path: {work_dir.resolve()}\n\ntrain: images/train\nval: images/val\n\nnames:\n  0: kurgan\n",
    encoding="utf-8",
)

metadata_path = work_dir / "metadata.csv"
meta = pd.read_csv(metadata_path)
images = meta.drop_duplicates("image")
boxes = meta[meta["class_name"].notna()]

print("Dataset:", work_dir)
print("Images:", len(images))
print("Positive images:", int(images["is_positive"].sum()))
print("BBox:", int(len(boxes)))
print(images.groupby(["split", "is_positive"]).size())


## 5. Locate best.pt


In [ ]:
def find_best_weights() -> Path:
    if WEIGHTS_INPUT_PATH.exists():
        return WEIGHTS_INPUT_PATH
    candidates = sorted(KAGGLE_INPUT_ROOT.rglob("best.pt"))
    if not candidates:
        raise FileNotFoundError("Attach the trained YOLOv8n v3b best.pt as a Kaggle input.")
    preferred = [
        p for p in candidates
        if "v3b" in str(p).lower() or "baseline" in str(p).lower() or "yolov8n" in str(p).lower()
    ]
    selected = preferred[0] if preferred else candidates[0]
    return selected

source_weights = find_best_weights()
weights_path = WEIGHTS_WORK_DIR / "v3b_yolov8n_best.pt"
shutil.copy2(source_weights, weights_path)
print("Source weights:", source_weights)
print("Working weights:", weights_path)
print("Size MB:", round(weights_path.stat().st_size / (1024 * 1024), 2))


## 6. Run Threshold Sweep


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(PROJECT_DIR / "scripts" / "sweep_v3b_thresholds.py"),
    "--metadata",
    str(metadata_path),
    "--weights",
    str(weights_path),
    "--out-dir",
    str(ANALYSIS_DIR),
    "--imgsz",
    str(IMGSZ),
    "--device",
    str(DEVICE),
    "--max-visual-images",
    "25",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=str(PROJECT_DIR), check=True)


## 7. Inspect Metrics


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

confidence = pd.read_csv(ANALYSIS_DIR / "confidence_sweep_metrics.csv")
nms = pd.read_csv(ANALYSIS_DIR / "nms_sweep_metrics.csv")

display(Markdown("### Confidence sweep"))
display(confidence)

display(Markdown("### NMS sweep"))
display(nms)

report_path = ANALYSIS_DIR / "threshold_sweep_report.md"
display(Markdown(report_path.read_text(encoding="utf-8")))


## 8. Visual Contact Sheets


In [ ]:
from IPython.display import Image as IPImage, display

for conf in ["0.25", "0.10", "0.05", "0.01"]:
    for kind in ["predictions", "false_positives", "false_negatives"]:
        path = ANALYSIS_DIR / f"conf_{conf}_{kind}.jpg"
        if path.exists():
            print(path.name)
            display(IPImage(filename=str(path)))


## 9. Archive Results


In [ ]:
import zipfile
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_path = Path("/kaggle/working") / f"yolo_v3b_threshold_sweep_{timestamp}.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in ANALYSIS_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(OUTPUT_ROOT))

print("Archive:", archive_path)
print("Archive size MB:", round(archive_path.stat().st_size / (1024 * 1024), 2))
